# Liu2024 S-JEPA Pretraining — 500 Hz Option B, with SSL-quality monitoring

This is the seconds-based ("Option B") native-500 Hz self-supervised pretraining: a self-contained
S-JEPA (per-channel `Conv1d+GELU+GroupNorm` encoder with seconds-scaled kernels, finite spatial+temporal
positional encoding, Transformer context model, EMA teacher, masked-latent prediction). The SSL model,
masking, EMA, objective, and checkpoint export are carried **verbatim** from the version you already ran
(they were reviewed and are correct); rewriting working PyTorch that can't be unit-tested here would only
add risk.

**What's new and why.** The original training signal was only the masked-latent L1 loss, which can keep
falling even if the representation **collapses** (predicts the mean). So a chance downstream result is
ambiguous: a collapsed encoder and a healthy encoder whose features simply don't transfer look identical.
This notebook adds two monitors evaluated periodically during training:

1. **Collapse metrics** on the embeddings — per-dim standard deviation, effective rank (participation
   ratio), and mean off-diagonal correlation. If std → 0 / rank → 1, the encoder is collapsing.
2. **Linear-probe** — on *known-decodable* subjects (from your permutation test), freeze the current
   encoder, mean-pool context tokens, and run within-subject CV logistic regression. The classical
   ceiling on these subjects is ~75%. If the probe stays at chance throughout while collapse metrics are
   healthy, the representation isn't capturing MI; if the probe climbs toward ~70%, the encoder *is*
   learning real structure and any chance downstream on subjects 41–50 is a transfer/data effect, not a
   pretraining bug.

> Note: the original recommended the `without_chans` export for downstream. For **same-montage** downstream
> (our case) use `with_chans` — it keeps the learned spatial positional encoding. Both are still saved.

## 1. Imports (carried) + probe deps

In [ ]:
import os
import re
import sys
import json
import math
import random
import hashlib
import platform
from copy import deepcopy
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from scipy.io import loadmat
import mne

# NOTE: braindecode.models.SignalJEPA intentionally NOT used — see root-cause note above.

mne.set_log_level("WARNING")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

print("Imports loaded")
print("Python:", sys.version)
print("Platform:", platform.platform())
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score


## 2. Configuration (carried verbatim)

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-sjepa-500hz-optionB-pretraining"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "liu2024_sjepa_500hz_optionB_pretraining",
    "config_note": "Native 500 Hz Liu2024 S-JEPA SSL pretraining with seconds-scaled temporal encoder.",

    # Subject split: downstream subjects remain unseen during SSL pretraining.
    "train_subject_ids": list(range(1, 31)),
    "val_subject_ids": list(range(31, 41)),
    "downstream_only_subject_ids": list(range(41, 51)),

    # Fixed Liu2024 preprocessing for this experiment.
    "sfreq": 500.0,
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",
    "reference_mode": "average",
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",

    # Pretraining windows.
    # Recommended: full_trial uses all 8 s available in Liu2024 for SSL context.
    # Alternative: fixed_crop with pretrain_start_s=1.5 and pretrain_window_s=4.2 aligns exactly to downstream.
    "pretrain_window_mode": "full_trial",     # full_trial, fixed_crop, sliding
    "pretrain_start_s": 0.0,
    "pretrain_window_s": 8.0,
    "sliding_stop_s": 8.0,
    "sliding_stride_s": 1.0,

    # Option B seconds-scaled S-JEPA local encoder.
    "conv_spec_mode": "seconds_scaled_500hz",
    "first_kernel_s": 0.25,
    "first_stride_s_target": 1.0 / 16.0,       # default S-JEPA first stride at 128 Hz is 8 samples = 0.0625 s
    "first_stride_rounding": "floor",          # floor gives stride 31 samples; total token stride ~= 0.992 s
    "later_kernel_samples": 2,
    "later_stride_samples": 2,

    # Masking objective.
    "mask_diameter_percent": 60.0,
    "predictor_n_layers": 4,
    "predictor_nhead": 8,
    "predictor_dim_feedforward": 256,
    "ema_decay": 0.996,

    # Context (target/online) transformer encoder.
    "context_n_layers": 4,
    "context_nhead": 8,
    "context_dim_feedforward": 256,
    "context_dropout": 0.0,

    # Training.
    "batch_size": 8,
    "n_epochs": 300,
    "early_stopping_patience": 30,
    "learning_rate": 1e-4,   # 1e-4 is safer for the transformer context encoder than 1e-3
    "weight_decay": 1e-4,
    "num_workers": 0,

    # Runtime.
    "device": "auto",
    "seed": 2026,
}

### 2b. Monitoring configuration (new)

In [ ]:
# Known-decodable subjects from the permutation test; only those present in train+val are probed.
# (Probing subjects the SSL pretrained on, WITHOUT their labels, is a fair representation-quality test:
#  can a linear head read MI out of the frozen features on subjects where we KNOW the signal exists?)
CONFIG["probe_subjects"] = [7, 22, 23, 28, 40]
CONFIG["probe_every"] = 5                 # run the probe/collapse check every N epochs (and at first/last)
CONFIG["probe_cv_folds"] = 5
CONFIG["probe_window_start_s"] = 1.5      # MI window for the probe (matches downstream)
CONFIG["probe_window_s"] = 4.2
CONFIG["probe_batch"] = 32
CONFIG["collapse_warn_std"] = 1e-3        # warn if mean per-dim std falls below this
# For same-montage downstream, prefer the with-chans export:
CONFIG["recommended_downstream_export"] = "with_chans"
print("probe subjects (will be intersected with loaded subjects):", CONFIG["probe_subjects"])


## 3. Run setup + seconds-based geometry (carried verbatim)

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def resolve_device(requested="auto"):
    requested = str(requested).lower()
    if requested == "cpu":
        return torch.device("cpu")
    if requested == "cuda":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def create_run_id(config):
    stamp = datetime.now().strftime("%Y%m%d_%H%M")
    digest = hashlib.md5(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{stamp}_500hz_{digest}"


def make_seconds_scaled_conv_spec(config):
    """Create a 500 Hz conv spec whose real-time behavior matches S-JEPA.

    Default S-JEPA at 128 Hz uses:
      (8, 32, 8), then four (kernel=2, stride=2) layers.
    That means:
      first kernel = 32/128 = 0.25 s
      total stride = 8*2*2*2*2 = 128 samples = 1.0 s
      receptive field = 152/128 = 1.1875 s

    At 500 Hz, we use first kernel ~= 125 samples and first stride ~= 31 samples.
    Total stride = 31*16 = 496 samples = 0.992 s.
    Receptive field = 590 samples = 1.18 s.
    """
    sfreq = float(config["sfreq"])
    k1 = int(round(float(config["first_kernel_s"]) * sfreq))
    raw_stride = float(config["first_stride_s_target"]) * sfreq
    if config.get("first_stride_rounding", "floor") == "floor":
        s1 = int(math.floor(raw_stride))
    elif config.get("first_stride_rounding") == "ceil":
        s1 = int(math.ceil(raw_stride))
    else:
        s1 = int(round(raw_stride))
    s1 = max(1, s1)
    k_later = int(config["later_kernel_samples"])
    s_later = int(config["later_stride_samples"])
    spec = (
        (8, k1, s1),
        (16, k_later, s_later),
        (32, k_later, s_later),
        (64, k_later, s_later),
        (64, k_later, s_later),
    )
    return spec


def conv_geometry(conv_spec, n_times, sfreq):
    out = int(n_times)
    total_stride = 1
    receptive = 1
    running_stride = 1
    for _, kernel, stride in conv_spec:
        out = (out - int(kernel)) // int(stride) + 1
        receptive = receptive + (int(kernel) - 1) * running_stride
        running_stride *= int(stride)
        total_stride *= int(stride)
    return {
        "n_tokens_per_channel": int(out),
        "total_stride_samples": int(total_stride),
        "token_sfreq_hz": float(sfreq) / float(total_stride),
        "token_stride_s": float(total_stride) / float(sfreq),
        "receptive_field_samples": int(receptive),
        "receptive_field_s": float(receptive) / float(sfreq),
    }


set_seed(CONFIG["seed"])
DEVICE = resolve_device(CONFIG["device"])
RUN_ID = create_run_id(CONFIG)
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CONV_LAYERS_SPEC = make_seconds_scaled_conv_spec(CONFIG)

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Artifact dir:", ARTIFACT_DIR)
print("Device:", DEVICE)
print("Conv spec:", CONV_LAYERS_SPEC)

## 4. Channel metadata + finite coordinates (carried verbatim)

In [ ]:
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17  # CPz source reference in Liu2024 source MAT
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CH_NAMES = [SOURCE_EEG_CHANNEL_NAMES_30[i] for i in SOURCE_EEG_CHANNEL_INDICES_29]
MONTAGE_ALIAS = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}

def build_channel_coordinates(ch_names):
    """Return (chs_info, xyz_meters, ch_pos_norm, audit_df). Raises on any missing/degenerate coord."""
    montage = mne.channels.make_standard_montage("standard_1020")
    ch_pos = montage.get_positions()["ch_pos"]
    info = mne.create_info(ch_names=list(ch_names), sfreq=float(CONFIG["sfreq"]), ch_types="eeg")
    xyz = np.zeros((len(ch_names), 3), dtype=np.float64)
    used = []
    for k, ch in enumerate(info["chs"]):
        name = ch["ch_name"]
        lookup = MONTAGE_ALIAS.get(name, name)
        if lookup not in ch_pos:
            raise RuntimeError(f"Missing montage coordinate for '{name}' (lookup '{lookup}'). "
                               f"Add an alias in MONTAGE_ALIAS.")
        pos = np.asarray(ch_pos[lookup], dtype=np.float64)
        if not np.isfinite(pos).all():
            raise RuntimeError(f"Non-finite montage coordinate for '{name}'.")
        loc = np.zeros(12, dtype=float); loc[:3] = pos
        ch["loc"][:] = loc
        xyz[k] = pos
        used.append(lookup)

    # finite / non-degenerate checks
    if not np.isfinite(xyz).all():
        raise RuntimeError("Non-finite channel coordinates after assembly.")
    axis_range = xyz.max(0) - xyz.min(0)
    if np.any(axis_range <= 1e-8):
        raise RuntimeError(f"Degenerate coordinate axis (range={axis_range}). Montage not 3-D.")
    # duplicate-position guard (two channels at the same point would break spatial encoding)
    from scipy.spatial.distance import pdist
    if pdist(xyz).min() <= 1e-9:
        raise RuntimeError("Two channels share identical coordinates; check names/aliases.")

    # normalized coordinates for the positional encoder: center + scale to unit ball, eps-safe
    center = xyz.mean(0)
    centered = xyz - center
    radius = float(np.linalg.norm(centered, axis=1).max())
    ch_pos_norm = (centered / (radius + 1e-8)).astype(np.float32)
    if not np.isfinite(ch_pos_norm).all():
        raise RuntimeError("Non-finite normalized coordinates (should be impossible with eps).")

    audit = pd.DataFrame({
        "ch_name": list(ch_names), "montage_lookup": used,
        "x_m": xyz[:, 0].round(4), "y_m": xyz[:, 1].round(4), "z_m": xyz[:, 2].round(4),
        "nx": ch_pos_norm[:, 0].round(3), "ny": ch_pos_norm[:, 1].round(3), "nz": ch_pos_norm[:, 2].round(3),
        "norm": np.linalg.norm(ch_pos_norm, axis=1).round(3),
    })
    return info["chs"], xyz, ch_pos_norm, audit

CHS_INFO, CH_XYZ_M, CH_POS_NORM, COORD_AUDIT = build_channel_coordinates(CH_NAMES)
CH_POSITIONS = torch.tensor(CH_XYZ_M.astype(np.float32))        # meters, for mask-sampler distances
CH_POS_NORM_T = torch.tensor(CH_POS_NORM)                        # normalized, for the pos encoder

print(f"Channels: {len(CH_NAMES)}")
print(CH_NAMES)
print("\nCoordinate audit (normalized coords are finite, |norm|<=1):")
try:
    from IPython.display import display; display(COORD_AUDIT)
except Exception:
    print(COORD_AUDIT.to_string(index=False))
assert np.isfinite(CH_POS_NORM).all()
print(f"\nAll {len(CH_NAMES)} channel coordinates finite; max |norm| = {np.linalg.norm(CH_POS_NORM,axis=1).max():.3f}")


## 5. Load + preprocess (carried verbatim; float64-correct)

In [ ]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []


def subject_id_from_path(path):
    text = str(path)
    match = re.search(r"sub[-_ ]?(\d{1,2})", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from {path}")


def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")


def _walk_mat_object(obj, prefix=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        if obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        else:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")


def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if any(token in lname for token in ["rawdata", "raw", "data"]):
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    return score


def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    score = 0
    if "label" in lname or "class" in lname:
        score += 10
    if flat.size in (39, 40):
        score += 5
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    if unique and unique.issubset({"0", "1", "2", "1.0", "2.0", "0.0"}):
        score += 3
    return score


def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []
    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]
    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)
    if arr.shape[1] < 30 or arr.shape[2] < 3000:
        raise ValueError(f"Could not normalize to trials x channels x samples, got {arr.shape}")
    return arr


def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    raw_candidates = []
    label_candidates = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            arr = np.asarray(value)
            if arr.ndim == 3:
                raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
            flat = arr.ravel()
            if flat.size in (39, 40):
                label_candidates.append((_score_label_candidate(name, arr), name, arr))
    if not raw_candidates:
        raise RuntimeError(f"No rawdata candidate found in {path}")
    raw_candidates.sort(key=lambda x: x[0], reverse=True)
    label_candidates.sort(key=lambda x: x[0], reverse=True)
    raw_name, raw = raw_candidates[0][1], raw_candidates[0][2]
    if not label_candidates:
        raise RuntimeError(f"No label candidate found in {path}")
    label_name, labels = label_candidates[0][1], label_candidates[0][2]
    raw = _normalize_rawdata_shape(raw, labels)
    labels = np.asarray(labels).ravel().astype(int)
    if labels.min() == 1:
        labels = labels - 1
    if raw.shape[0] != labels.size:
        raise RuntimeError(f"Trial/label mismatch for {path}: raw={raw.shape}, labels={labels.shape}")
    return raw, labels.astype(np.int64), raw_name, label_name


def preprocess_subject(rawdata, labels, subject_id):
    """Return preprocessed full 8 s trials as microvolts, shape trials x 29 x 4000.

    MNE FIR filtering expects float64 input in recent versions.  Keep the
    filtering/reference path in float64, then cast back to float32 only after
    preprocessing so the training tensors remain compact.
    """
    X = np.asarray(rawdata[:, SOURCE_EEG_CHANNEL_INDICES_29, :], dtype=np.float64)
    y = np.asarray(labels, dtype=np.int64)

    # Convert to volts for MNE filtering, average-reference before filter, then return microvolts.
    X_volts = X * 1e-6 if CONFIG["source_unit"] == "microvolts" else X
    X_volts = np.asarray(X_volts, dtype=np.float64, order="C")
    if CONFIG["reference_mode"] == "average":
        X_volts = X_volts - X_volts.mean(axis=1, keepdims=True)
        X_volts = np.asarray(X_volts, dtype=np.float64, order="C")
    X_volts = mne.filter.filter_data(
        X_volts,
        sfreq=float(CONFIG["sfreq"]),
        l_freq=float(CONFIG["filter_low"]),
        h_freq=float(CONFIG["filter_high"]),
        method=CONFIG["filter_method"],
        phase="zero",
        fir_design="firwin",
        verbose=False,
    )
    X_uv = X_volts * 1e6 if CONFIG["final_model_unit"] == "microvolts" else X_volts
    return X_uv.astype(np.float32, copy=False), y


def make_pretraining_windows(X_full_trials, y):
    sfreq = float(CONFIG["sfreq"])
    n_times = X_full_trials.shape[-1]
    mode = CONFIG["pretrain_window_mode"]
    windows = []
    labels = []

    if mode == "full_trial":
        start = 0
        length = n_times
        windows = [X_full_trials]
        labels = [y]
    elif mode == "fixed_crop":
        start = int(round(float(CONFIG["pretrain_start_s"]) * sfreq))
        length = int(round(float(CONFIG["pretrain_window_s"]) * sfreq))
        stop = start + length
        if stop > n_times:
            raise RuntimeError(f"fixed_crop [{start}:{stop}] exceeds trial length {n_times}")
        windows = [X_full_trials[:, :, start:stop]]
        labels = [y]
    elif mode == "sliding":
        length = int(round(float(CONFIG["pretrain_window_s"]) * sfreq))
        start0 = int(round(float(CONFIG["pretrain_start_s"]) * sfreq))
        stop_limit = int(round(float(CONFIG["sliding_stop_s"]) * sfreq))
        stride = int(round(float(CONFIG["sliding_stride_s"]) * sfreq))
        for start in range(start0, stop_limit - length + 1, stride):
            windows.append(X_full_trials[:, :, start:start + length])
            labels.append(y)
    else:
        raise ValueError("pretrain_window_mode must be full_trial, fixed_crop, or sliding")

    Xw = np.concatenate(windows, axis=0).astype(np.float32)
    yw = np.concatenate(labels, axis=0).astype(np.int64)
    return Xw, yw


class ArrayDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.zeros(len(self.X), dtype=np.int64) if y is None else np.asarray(y, dtype=np.int64)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), int(self.y[idx])

## 6. Build SSL datasets — subjects 1–30 train, 31–40 val, 41–50 excluded (carried verbatim)

In [ ]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
mat_files = find_source_mat_files(SOURCE_EXTRACT_DIR)
if not mat_files:
    raise RuntimeError(f"No .mat files found under {SOURCE_EXTRACT_DIR}")

all_subject_paths = {subject_id_from_path(path): path for path in mat_files}
required = set(CONFIG["train_subject_ids"] + CONFIG["val_subject_ids"] + CONFIG["downstream_only_subject_ids"])
missing = sorted(required - set(all_subject_paths))
if missing:
    raise RuntimeError(f"Missing expected subject files: {missing}")

SUBJECT_WINDOWS = {}
rows = []
for sid in sorted(set(CONFIG["train_subject_ids"] + CONFIG["val_subject_ids"])):
    raw, labels, raw_field, label_field = load_subject_mat(all_subject_paths[sid])
    X_trials, y = preprocess_subject(raw, labels, sid)
    Xw, yw = make_pretraining_windows(X_trials, y)
    SUBJECT_WINDOWS[str(sid)] = ArrayDataset(Xw, yw)
    rows.append({
        "subject_id": sid,
        "raw_shape": list(raw.shape),
        "preprocessed_trial_shape": list(X_trials.shape),
        "window_shape": list(Xw.shape),
        "n_windows": int(len(Xw)),
        "class_0": int((yw == 0).sum()),
        "class_1": int((yw == 1).sum()),
        "raw_field": raw_field,
        "label_field": label_field,
    })

inventory = pd.DataFrame(rows)
inventory.to_csv(ARTIFACT_DIR / "subject_window_inventory.csv", index=False)
display(inventory.head())
print("Total SSL train windows:", sum(len(SUBJECT_WINDOWS[str(s)]) for s in CONFIG["train_subject_ids"]))
print("Total SSL val windows:", sum(len(SUBJECT_WINDOWS[str(s)]) for s in CONFIG["val_subject_ids"]))

WINDOW_SAMPLES = int(inventory.iloc[0]["window_shape"][-1])
INPUT_WINDOW_SECONDS = WINDOW_SAMPLES / float(CONFIG["sfreq"])
GEOMETRY = conv_geometry(CONV_LAYERS_SPEC, WINDOW_SAMPLES, float(CONFIG["sfreq"]))
print("Window samples:", WINDOW_SAMPLES)
print("Input seconds:", INPUT_WINDOW_SECONDS)
print("Token geometry:", GEOMETRY)
if GEOMETRY["n_tokens_per_channel"] < 2:
    raise RuntimeError("Too few tokens per channel. Increase pretrain_window_s or use full_trial.")

## 7. Self-contained S-JEPA model + instantiation (carried VERBATIM)

In [ ]:
def assert_finite(name, tensor):
    if not torch.isfinite(tensor).all():
        bad = int((~torch.isfinite(tensor)).sum().item())
        raise RuntimeError(f"Non-finite tensor detected: {name}; count={bad}")


class ConvFeatureEncoder(nn.Module):
    """Per-channel temporal conv stack (S-JEPA local encoder). (B,C,T) -> (B,C,n_tok,D)."""
    def __init__(self, conv_spec):
        super().__init__()
        blocks, in_ch = [], 1
        for out_ch, k, s in conv_spec:
            blocks += [nn.Conv1d(in_ch, int(out_ch), kernel_size=int(k), stride=int(s)),
                       nn.GELU(), nn.GroupNorm(1, int(out_ch))]   # GroupNorm: batch-size-independent, stable
            in_ch = int(out_ch)
        self.net = nn.Sequential(*blocks); self.out_dim = in_ch
    def forward(self, x):
        B, C, T = x.shape
        h = self.net(x.reshape(B * C, 1, T))         # (B*C, D, n_tok)
        D, n_tok = h.shape[1], h.shape[-1]
        return h.permute(0, 2, 1).reshape(B, C, n_tok, D)


class PositionalEncoder(nn.Module):
    """Finite-by-construction PE: spatial = MLP(normalized xyz); temporal = bounded sinusoid."""
    def __init__(self, d_model, ch_pos_norm, max_tokens=8192):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for sinusoidal PE"
        self.d_model = d_model
        self.register_buffer("ch_pos", torch.as_tensor(ch_pos_norm, dtype=torch.float32))
        self.spatial_mlp = nn.Sequential(nn.Linear(3, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        pe = torch.zeros(max_tokens, d_model)
        pos = torch.arange(max_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("temporal_pe", pe)
    def spatial(self):                 # (C, D)
        return self.spatial_mlp(self.ch_pos)
    def temporal(self, n_tok):         # (n_tok, D)
        return self.temporal_pe[:n_tok]
    def forward(self, n_chans, n_tok):  # (C*n_tok, D), channel-major to match feature flattening
        sp = self.spatial()                       # (C, D)
        tp = self.temporal(n_tok)                 # (n_tok, D)
        return (sp[:, None, :] + tp[None, :, :]).reshape(n_chans * n_tok, self.d_model)


class SJEPABackbone(nn.Module):
    """feature_encoder -> (+ positional) -> context_encoder. EMA-copied to form the target encoder."""
    def __init__(self, conv_spec, ch_pos_norm, n_layers, nhead, dim_ff, dropout):
        super().__init__()
        self.feature_encoder = ConvFeatureEncoder(conv_spec)
        D = self.feature_encoder.out_dim
        self.pos_encoder = PositionalEncoder(D, ch_pos_norm)
        layer = nn.TransformerEncoderLayer(d_model=D, nhead=nhead, dim_feedforward=dim_ff,
                                           dropout=dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.context_encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.d_model = D
    def tokens_pe(self, x):
        local = self.feature_encoder(x)                 # (B,C,n_tok,D)
        B, C, n_tok, D = local.shape
        pe = self.pos_encoder(C, n_tok)                 # (C*n_tok, D)
        return local.reshape(B, C * n_tok, D), pe, (B, C, n_tok, D)


class MaskedTokenPredictor(nn.Module):
    def __init__(self, d_model=64, nhead=8, num_layers=4, dim_feedforward=256, dropout=0.0):
        super().__init__()
        layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                           dropout=dropout, batch_first=True, activation="gelu", norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
    def forward(self, context_tokens, masked_queries):
        return self.decoder(tgt=masked_queries, memory=context_tokens)


class RandomSpatialBlockMaskSampler:
    """Mask a spatial block of channels (all their tokens). Token-major flatten matches the encoder."""
    def __init__(self, ch_positions, ch_names, mask_diameter_percent, n_tok_per_channel):
        self.ch_positions = torch.as_tensor(ch_positions, dtype=torch.float32)
        self.ch_names = list(ch_names); self.n_channels = len(ch_names)
        self.n_tok_per_channel = int(n_tok_per_channel)
        dists = torch.cdist(self.ch_positions, self.ch_positions)
        self.head_diameter = float(dists.max().item())
        self.mask_diameter_percent = float(mask_diameter_percent)
        self.mask_radius = (self.mask_diameter_percent / 100.0) * self.head_diameter / 2.0
        self.distances = dists
    def sample_one(self, device):
        center = random.randrange(self.n_channels)
        mask_ch = self.distances[center] <= self.mask_radius
        if bool(mask_ch.all()):
            mask_ch[int(torch.argmax(self.distances[center]).item())] = False
        if not bool(mask_ch.any()):
            mask_ch[center] = True
        mask_tok = mask_ch.repeat_interleave(self.n_tok_per_channel).to(device)
        return mask_ch.to(device), mask_tok, center
    def sample(self, batch_size, device):
        mcs, mts, cs = [], [], []
        for _ in range(batch_size):
            mc, mt, c = self.sample_one(device); mcs.append(mc); mts.append(mt); cs.append(c)
        mask_ch = torch.stack(mcs, 0); mask_tok = torch.stack(mts, 0)
        info = {"center_indices": cs,
                "masked_channel_counts": mask_ch.sum(1).detach().cpu().numpy().astype(int).tolist(),
                "masked_token_counts": mask_tok.sum(1).detach().cpu().numpy().astype(int).tolist()}
        return mask_ch, mask_tok, info


# ---- build student (online), teacher (EMA target), predictor, mask token ----
STUDENT = SJEPABackbone(CONV_LAYERS_SPEC, CH_POS_NORM,
                        n_layers=int(CONFIG["context_n_layers"]), nhead=int(CONFIG["context_nhead"]),
                        dim_ff=int(CONFIG["context_dim_feedforward"]), dropout=float(CONFIG["context_dropout"])).to(DEVICE)
TEACHER = deepcopy(STUDENT).to(DEVICE); TEACHER.eval()
for p in TEACHER.parameters():
    p.requires_grad = False

D_MODEL = STUDENT.d_model
PREDICTOR = MaskedTokenPredictor(d_model=D_MODEL, nhead=int(CONFIG["predictor_nhead"]),
                                 num_layers=int(CONFIG["predictor_n_layers"]),
                                 dim_feedforward=int(CONFIG["predictor_dim_feedforward"])).to(DEVICE)
MASK_TOKEN = nn.Parameter(torch.zeros(1, 1, D_MODEL, device=DEVICE))

# ---- probe + finite checks, ISOLATING spatial vs temporal positional encoding ----
probe_x, _ = SUBJECT_WINDOWS[str(CONFIG["val_subject_ids"][0])][0]
probe_x = probe_x.unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    probe_local = STUDENT.feature_encoder(probe_x);                 assert_finite("local", probe_local)
    _, C_, ntk_, D_ = probe_local.shape
    sp = STUDENT.pos_encoder.spatial();                            assert_finite("spatial_pe", sp)
    tp = STUDENT.pos_encoder.temporal(ntk_);                       assert_finite("temporal_pe", tp)
    pe = STUDENT.pos_encoder(C_, ntk_);                            assert_finite("combined_pe", pe)
    full = probe_local.reshape(1, C_ * ntk_, D_) + pe.unsqueeze(0); assert_finite("full", full)
    ctx = STUDENT.context_encoder(full);                          assert_finite("context", ctx)

ACTUAL_TOTAL_TOKENS = int(C_ * ntk_); ACTUAL_EMB_DIM = int(D_); ACTUAL_N_TOK_PER_CHANNEL = int(ntk_)
if ACTUAL_TOTAL_TOKENS % len(CH_NAMES) != 0:
    raise RuntimeError(f"Token count {ACTUAL_TOTAL_TOKENS} not divisible by n_channels={len(CH_NAMES)}")

MASK_SAMPLER = RandomSpatialBlockMaskSampler(CH_POSITIONS, CH_NAMES,
                                             mask_diameter_percent=CONFIG["mask_diameter_percent"],
                                             n_tok_per_channel=ACTUAL_N_TOK_PER_CHANNEL)

print("spatial / temporal / combined positional encodings all finite ✓  (root cause resolved)")
print("Local feature shape:", tuple(probe_local.shape))
print("Tokens/channel:", ACTUAL_N_TOK_PER_CHANNEL, "| total tokens:", ACTUAL_TOTAL_TOKENS, "| emb dim:", ACTUAL_EMB_DIM)
print("Mask radius:", MASK_SAMPLER.mask_radius)


## 8. SSL objective, EMA, optimizer, epoch runner (carried VERBATIM)

In [ ]:
@torch.no_grad()
def ema_update(student, teacher, decay):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data.mul_(decay).add_(ps.data, alpha=1.0 - decay)


def compute_single_sample_loss(x_single):
    assert_finite("input", x_single)
    _, mask_tok_b, info = MASK_SAMPLER.sample(1, device=DEVICE)
    mask_tok = mask_tok_b[0]                                   # (N,) bool over C*n_tok

    with torch.no_grad():
        t_tokens, t_pe, (B, C, ntk, D) = TEACHER.tokens_pe(x_single)
        t_ctx = TEACHER.context_encoder(t_tokens + t_pe.unsqueeze(0))
        target = t_ctx[:, mask_tok, :]
        assert_finite("target", target)

    s_tokens, s_pe, _ = STUDENT.tokens_pe(x_single)
    assert_finite("student_local", s_tokens); assert_finite("student_pe", s_pe)
    s_full = s_tokens + s_pe.unsqueeze(0)
    visible = s_full[:, ~mask_tok, :]
    s_ctx = STUDENT.context_encoder(visible)
    assert_finite("student_context", s_ctx)

    n_masked = int(mask_tok.sum().item())
    queries = MASK_TOKEN.expand(B, n_masked, D) + s_pe[mask_tok].unsqueeze(0)
    pred = PREDICTOR(s_ctx, queries)
    assert_finite("predictor", pred)

    loss = F.smooth_l1_loss(pred, target)
    assert_finite("loss", loss)
    return loss, info


def process_batch(X, training=True):
    X = X.float().to(DEVICE)
    (STUDENT.train(), PREDICTOR.train()) if training else (STUDENT.eval(), PREDICTOR.eval())
    TEACHER.eval()
    losses, mch, mtok = [], [], []
    for i in range(X.shape[0]):
        loss, info = compute_single_sample_loss(X[i:i + 1])
        losses.append(loss); mch.append(info["masked_channel_counts"][0]); mtok.append(info["masked_token_counts"][0])
    return torch.stack(losses).mean(), float(np.mean(mch)), float(np.mean(mtok))


def make_loader(subject_ids, shuffle):
    ds = ConcatDataset([SUBJECT_WINDOWS[str(int(s))] for s in subject_ids])
    return DataLoader(ds, batch_size=int(CONFIG["batch_size"]), shuffle=shuffle,
                      num_workers=int(CONFIG["num_workers"]), drop_last=False)


OPTIMIZER = torch.optim.AdamW(
    list(STUDENT.parameters()) + list(PREDICTOR.parameters()) + [MASK_TOKEN],
    lr=float(CONFIG["learning_rate"]), weight_decay=float(CONFIG["weight_decay"]))


def run_epoch(subject_ids, training, epoch):
    loader = make_loader(subject_ids, shuffle=training)
    total_loss, total_n, mch, mtok = 0.0, 0, [], []
    for X, _ in loader:
        if training:
            OPTIMIZER.zero_grad(set_to_none=True)
            loss, mc, mt = process_batch(X, training=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(STUDENT.parameters()) + list(PREDICTOR.parameters()) + [MASK_TOKEN], 5.0)
            OPTIMIZER.step()
            ema_update(STUDENT, TEACHER, float(CONFIG["ema_decay"]))
        else:
            with torch.no_grad():
                loss, mc, mt = process_batch(X, training=False)
        n = int(X.shape[0]); total_loss += float(loss.item()) * n; total_n += n
        mch.append(mc); mtok.append(mt)
    return {"loss": total_loss / max(1, total_n), "n_examples": total_n,
            "masked_channels_mean": float(np.mean(mch)), "masked_tokens_mean": float(np.mean(mtok))}


## 9. Checkpoint + metadata export (carried VERBATIM)

In [ ]:
def backbone_export_metadata():
    return {
        "sfreq": float(CONFIG["sfreq"]),
        "input_window_seconds": float(INPUT_WINDOW_SECONDS),
        "window_samples": int(WINDOW_SAMPLES),
        "conv_layers_spec": [list(x) for x in CONV_LAYERS_SPEC],
        "conv_geometry": GEOMETRY,
        "actual_token_geometry": {
            "n_channels": len(CH_NAMES),
            "n_tok_per_channel": int(ACTUAL_N_TOK_PER_CHANNEL),
            "emb_dim": int(ACTUAL_EMB_DIM),
            "total_tokens": int(ACTUAL_TOTAL_TOKENS),
        },
        "ch_names": CH_NAMES,
        "chs_info": CHS_INFO,
        "preprocessing_config": {
            "sfreq": float(CONFIG["sfreq"]),
            "reference_mode": CONFIG["reference_mode"],
            "filter_low": float(CONFIG["filter_low"]),
            "filter_high": float(CONFIG["filter_high"]),
            "filter_method": CONFIG["filter_method"],
            "pretrain_window_mode": CONFIG["pretrain_window_mode"],
            "pretrain_start_s": float(CONFIG["pretrain_start_s"]),
            "pretrain_window_s": float(CONFIG["pretrain_window_s"]),
        },
        "subject_split": {
            "train_subject_ids": CONFIG["train_subject_ids"],
            "val_subject_ids": CONFIG["val_subject_ids"],
            "downstream_only_subject_ids": CONFIG["downstream_only_subject_ids"],
        },
        "masking_config": {
            "mask_diameter_percent": float(CONFIG["mask_diameter_percent"]),
            "head_diameter": float(MASK_SAMPLER.head_diameter),
            "mask_radius": float(MASK_SAMPLER.mask_radius),
        },
    }


def strip_channel_specific_state_dict(state_dict):
    stripped = {}
    removed = []
    for key, value in state_dict.items():
        lk = str(key).lower()
        if "pos_encoder" in lk or "channel" in lk or "chan" in lk or "chs" in lk:
            removed.append(key)
        else:
            stripped[key] = value
    return stripped, removed


def save_checkpoint(tag, epoch, record, metrics):
    meta = backbone_export_metadata()
    checkpoint = {
        "epoch": int(epoch),
        "epoch_record": record,
        "metrics": metrics,
        "student_state_dict": STUDENT.state_dict(),
        "mask_token": MASK_TOKEN.detach().cpu(),
        "teacher_state_dict": TEACHER.state_dict(),
        "predictor_state_dict": PREDICTOR.state_dict(),
        "optimizer_state_dict": OPTIMIZER.state_dict(),
        "backbone_export_metadata": meta,
    }
    torch.save(checkpoint, ARTIFACT_DIR / f"checkpoint_{tag}.pt")

    full_payload = {
        "student_backbone_state_dict": STUDENT.state_dict(),
        "export_variant": "with_chans",
        **meta,
    }
    torch.save(full_payload, ARTIFACT_DIR / f"student_backbone_with_chans_{tag}.pt")
    torch.save(full_payload, ARTIFACT_DIR / f"student_backbone_{tag}.pt")

    stripped, removed = strip_channel_specific_state_dict(STUDENT.state_dict())
    nochan_payload = {
        "student_backbone_state_dict": stripped,
        "removed_channel_specific_keys": removed,
        "export_variant": "without_chans",
        **meta,
    }
    torch.save(nochan_payload, ARTIFACT_DIR / f"student_backbone_without_chans_{tag}.pt")

## 10. NEW — collapse metrics + linear-probe on known-decodable subjects

`embed_trials` mirrors the downstream forward exactly: `tokens, pe = backbone.tokens_pe(x)` →
`context_encoder(tokens + pe)` → mean-pool tokens. The probe builds a small labeled set once (cached)
and re-embeds it with the current student each time it runs.

In [ ]:
@torch.no_grad()
def embed_trials(backbone, X, batch=None):
    """Frozen pooled context embeddings for X (trials, C, T) -> (trials, D) numpy."""
    backbone.eval()
    batch = int(batch or CONFIG["probe_batch"])
    out = []
    for i in range(0, len(X), batch):
        xb = torch.as_tensor(np.asarray(X[i:i+batch]), dtype=torch.float32, device=DEVICE)
        tokens, pe, _ = backbone.tokens_pe(xb)
        ctx = backbone.context_encoder(tokens + pe.unsqueeze(0))
        out.append(ctx.mean(dim=1).float().cpu().numpy())
    return np.concatenate(out, 0)

def _crop_probe(Xf):
    s = int(round(CONFIG["probe_window_start_s"] * CONFIG["sfreq"]))
    n = int(round(CONFIG["probe_window_s"] * CONFIG["sfreq"]))
    return Xf[:, :, s:s+n]

# build the probe set ONCE (raw trials cached; embeddings recomputed each probe)
PROBE_SUBJECTS = [s for s in CONFIG["probe_subjects"]
                  if s in (CONFIG["train_subject_ids"] + CONFIG["val_subject_ids"]) and s in all_subject_paths]
PROBE_DATA = {}
for sid in PROBE_SUBJECTS:
    raw, labels, *_ = load_subject_mat(all_subject_paths[sid])
    Xf, y = preprocess_subject(raw, labels, sid)
    PROBE_DATA[sid] = (_crop_probe(Xf).astype(np.float32), y.astype(int))
print("probe subjects available:", PROBE_SUBJECTS)

def collapse_metrics(emb):
    """emb: (n, D). Returns per-dim std, effective rank (participation ratio), mean |off-diag corr|."""
    if len(emb) < 3:
        return {"emb_std": float("nan"), "effective_rank": float("nan"), "offdiag_corr": float("nan")}
    std = float(emb.std(axis=0).mean())
    cov = np.cov(emb, rowvar=False)
    ev = np.clip(np.linalg.eigvalsh(cov), 0, None)
    eff_rank = float((ev.sum() ** 2) / (np.square(ev).sum() + 1e-12))
    corr = np.corrcoef(emb, rowvar=False)
    d = corr.shape[0]; offdiag = corr[~np.eye(d, dtype=bool)]
    return {"emb_std": std, "effective_rank": eff_rank, "offdiag_corr": float(np.nanmean(np.abs(offdiag)))}

def linear_probe(backbone):
    """Within-subject CV logreg balanced accuracy on frozen embeddings of known-decodable subjects."""
    per_subj, all_emb = {}, []
    for sid, (X, y) in PROBE_DATA.items():
        emb = embed_trials(backbone, X); all_emb.append(emb)
        if len(np.unique(y)) < 2:
            continue
        skf = StratifiedKFold(n_splits=min(CONFIG["probe_cv_folds"], int(np.bincount(y).min())),
                              shuffle=True, random_state=int(CONFIG["seed"]))
        accs = []
        for tr, te in skf.split(emb, y):
            sc = StandardScaler().fit(emb[tr])
            clf = LogisticRegression(max_iter=2000).fit(sc.transform(emb[tr]), y[tr])
            accs.append(balanced_accuracy_score(y[te], clf.predict(sc.transform(emb[te]))))
        per_subj[sid] = float(np.mean(accs))
    coll = collapse_metrics(np.concatenate(all_emb, 0)) if all_emb else {}
    probe_mean = float(np.mean(list(per_subj.values()))) if per_subj else float("nan")
    return {"probe_mean_balacc": probe_mean, "probe_per_subject": per_subj, **coll}


## 11. Run pretraining with periodic monitoring (new orchestration)

In [ ]:
metrics, best_val, best_epoch, patience, stop_reason = [], float("inf"), -1, 0, "max_epochs_reached"
probe_every = int(CONFIG["probe_every"]); n_epochs = int(CONFIG["n_epochs"])

print("=" * 80)
print("500 Hz Option B S-JEPA pretraining (monitored)")
print("train:", CONFIG["train_subject_ids"], "| val:", CONFIG["val_subject_ids"])
print("probe (known-decodable):", PROBE_SUBJECTS, "| classical ceiling on these ~0.75")
print("conv spec:", CONV_LAYERS_SPEC, "| geometry:", GEOMETRY)
print("=" * 80)

for epoch in range(1, n_epochs + 1):
    train_m = run_epoch(CONFIG["train_subject_ids"], training=True, epoch=epoch)
    val_m = run_epoch(CONFIG["val_subject_ids"], training=False, epoch=epoch)
    record = {"epoch": int(epoch), "train_loss": float(train_m["loss"]), "val_loss": float(val_m["loss"]),
              "train_n": int(train_m["n_examples"]), "val_n": int(val_m["n_examples"]),
              "train_masked_tokens_mean": float(train_m["masked_tokens_mean"])}

    do_probe = PROBE_DATA and (epoch == 1 or epoch == n_epochs or epoch % probe_every == 0)
    if do_probe:
        pr = linear_probe(STUDENT)
        record.update({"probe_mean_balacc": pr["probe_mean_balacc"], "emb_std": pr["emb_std"],
                       "effective_rank": pr["effective_rank"], "offdiag_corr": pr["offdiag_corr"],
                       "probe_per_subject": pr["probe_per_subject"]})
        if pr["emb_std"] < CONFIG["collapse_warn_std"]:
            print(f"  [COLLAPSE WARNING] epoch {epoch}: embedding std {pr['emb_std']:.2e} ~ 0")

    metrics.append(record)
    pd.DataFrame(metrics).to_csv(ARTIFACT_DIR / "metrics.csv", index=False)
    with open(ARTIFACT_DIR / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2, default=str)

    if record["val_loss"] < best_val:
        best_val, best_epoch, patience = record["val_loss"], epoch, 0
        save_checkpoint("best", epoch, record, metrics)
    else:
        patience += 1
    save_checkpoint("latest", epoch, record, metrics)

    msg = (f"epoch={epoch:03d} train={record['train_loss']:.5f} val={record['val_loss']:.5f} "
           f"best={best_val:.5f}@{best_epoch} pat={patience}/{CONFIG['early_stopping_patience']}")
    if do_probe:
        msg += (f" | probe={record['probe_mean_balacc']:.3f} std={record['emb_std']:.3f} "
                f"effrank={record['effective_rank']:.1f}")
    print(msg)

    if patience >= int(CONFIG["early_stopping_patience"]):
        stop_reason = "early_stopping"; break

summary = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "best_epoch": int(best_epoch), "best_val_loss": float(best_val),
    "epochs_completed": int(len(metrics)), "stop_reason": stop_reason,
    "recommended_downstream_checkpoint": str(ARTIFACT_DIR / "student_backbone_with_chans_best.pt"),
    "probe_subjects": PROBE_SUBJECTS,
    "final_probe": next((m.get("probe_mean_balacc") for m in reversed(metrics) if "probe_mean_balacc" in m), None),
    "config": CONFIG, "conv_layers_spec": [list(x) for x in CONV_LAYERS_SPEC], "conv_geometry": GEOMETRY,
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("\nRecommended downstream checkpoint (same-montage):", summary["recommended_downstream_checkpoint"])
print("Final probe balanced accuracy on known-decodable subjects:", summary["final_probe"])


## 12. Diagnostics plot — loss, probe, and collapse

In [ ]:
import matplotlib.pyplot as plt
m = pd.DataFrame(metrics)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(m["epoch"], m["train_loss"], label="train")
axes[0].plot(m["epoch"], m["val_loss"], label="val")
axes[0].set_title("Masked-latent L1 loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

if "probe_mean_balacc" in m.columns:
    p = m.dropna(subset=["probe_mean_balacc"])
    axes[1].plot(p["epoch"], p["probe_mean_balacc"], "o-", color="#2c7fb8")
    axes[1].axhline(0.5, ls="--", c="grey", lw=1); axes[1].axhline(0.75, ls=":", c="green", lw=1)
    axes[1].set_ylim(0.4, 0.85); axes[1].set_title("Linear probe (known-decodable)\ngreen = classical ~0.75")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("balanced accuracy")
    if "effective_rank" in m.columns:
        axes[2].plot(p["epoch"], p["effective_rank"], "o-", color="#d95f02", label="effective rank")
        axes[2].set_title("Collapse: embedding effective rank"); axes[2].set_xlabel("epoch")
        axes[2].axhline(1.0, ls="--", c="red", lw=1, label="collapse"); axes[2].legend()
else:
    axes[1].set_title("probe disabled (no probe subjects)"); axes[2].set_title("")

fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "pretraining_diagnostics.png", dpi=160); plt.show()
print("Saved diagnostics ->", ARTIFACT_DIR / "pretraining_diagnostics.png")


## 13. How to read the monitors (decision guide)

- **Probe climbs toward ~0.70 on the known-decodable subjects** and effective rank stays well above 1:
  the encoder is learning genuine, decodable structure. A chance downstream on subjects 41–50 is then a
  transfer/data effect (those subjects are mostly non-decodable; the signal is subject-idiosyncratic) —
  *not* a pretraining bug. This is the expected outcome given everything found so far.
- **Probe stays at chance while effective rank collapses toward 1 / embedding std → 0:** the SSL
  collapsed — a real, fixable bug. Mitigations: lower EMA decay early, add a small variance/covariance
  (VICReg-style) penalty on the student embeddings, reduce predictor capacity, or increase mask difficulty.
- **Probe at chance but rank healthy:** the representation is rich but not MI-aligned — consistent with
  SSL learning nuisance structure rather than the rare motor-imagery contrast on this cohort.

Then run the matched downstream notebook (`liu2024_sjepa_500hz_optionB_downstream_custom.ipynb`) pointing
at `student_backbone_with_chans_best.pt`.